[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_8_Pipeline.ipynb)

# 08.8: A Complete Cleaning Pipeline

Every notebook so far has focused on one tool at a time: handling missing data, fixing types, normalizing strings, parsing dates. In practice you need all of them in sequence, and the order matters: you cannot parse a date you have not yet loaded, and you cannot fill a null in a column you have already dropped.

This notebook applies all of those tools to a dataset you have not seen before: a sample of Chicago 311 abandoned vehicle complaints. The goal is not just a cleaner DataFrame, but a single reusable function that takes the raw data and returns it ready for analysis.

In [1]:
import pandas as pd

## Step 0: Survey the damage

The first thing to do with any new dataset is load it and look at it without touching anything. `df.info()` in one call shows the row count, all column names, their dtypes, and how many non-null values each column has.

In [2]:
url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/refs/heads/master/data_311_Service_Requests_small.csv"
raw = pd.read_csv(url)
print(f"Shape: {raw.shape}")
raw.info()

Shape: (74999, 17)
<class 'pandas.DataFrame'>
RangeIndex: 74999 entries, 0 to 74998
Data columns (total 17 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   Creation Date                                           74999 non-null  str    
 1   Status                                                  74999 non-null  str    
 2   Completion Date                                         74999 non-null  str    
 3   Service Request Number                                  74999 non-null  str    
 4   Type of Service Request                                 74999 non-null  str    
 5   License Plate                                           72930 non-null  str    
 6   Vehicle Make/Model                                      71921 non-null  str    
 7   Vehicle Color                                           73508 non-null  str    
 8   Current Activity            

Scanning this output reveals several problems before any cleaning has happened:

- Column names have spaces, slashes, and a full question mark (`How Many Days Has the Vehicle Been Reported as Parked?`). These are awkward to reference in code.
- `Creation Date` and `Completion Date` are string columns (`object`), not datetimes.
- `SSA` shows far fewer non-null values than the total row count, suggesting heavy missingness.
- Several columns (`Current Activity`, `Most Recent Action`) appear to be mostly or entirely null.

Let's look at a few rows to understand the content.

In [3]:
raw.head(3)

,Creation Date,Status,Completion Date,Service Request Number,Type of Service Request,License Plate,Vehicle Make/Model,Vehicle Color,Current Activity,Most Recent Action,How Many Days Has the Vehicle Been Reported as Parked?,Street Address,ZIP Code,Ward,Police District,Community Area,SSA
0,01/01/2011,Completed,01/05/2011,11-00001976,Abandoned Vehicle Complaint,H924236,Ford,White,NaN,NaN,60.0,6059 S KOMENSKY AVE,60629.0,13.0,8.0,65.0,3.0
1,01/01/2011,Completed,01/05/2011,11-00002291,Abandoned Vehicle Complaint,810 LYB WISCONSIN PLATES,Mercury,Green,NaN,NaN,NaN,4651 S WASHTENAW AVE,60632.0,12.0,9.0,58.0,NaN
2,01/01/2011,Completed,01/05/2011,11-00002696,Abandoned Vehicle Complaint,368M783,Buick,Gold,NaN,NaN,10.0,6200 S MASSASOIT AVE,60638.0,13.0,8.0,64.0,NaN


Each row represents one abandoned vehicle complaint. The `Type of Service Request` column says `Abandoned Vehicle Complaint` for every visible row. The street address is in all-caps. Dates are in `MM/DD/YYYY` format. `Current Activity` and `Most Recent Action` are `NaN` for these rows. 

One more diagnostic before writing any cleaning code: check how many unique values each column has.

In [4]:
# Columns with only one unique value carry no information
nunique = raw.nunique()
print("Unique value counts per column:")
print(nunique.sort_values())

Unique value counts per column:
Status                                                        1
Type of Service Request                                       1
Current Activity                                              2
Most Recent Action                                            8
Vehicle Color                                                27
Police District                                              27
Ward                                                         51
SSA                                                          53
Community Area                                               78
Vehicle Make/Model                                          114
ZIP Code                                                    174
How Many Days Has the Vehicle Been Reported as Parked?      340
Completion Date                                            1073
Creation Date                                              1641
License Plate                                             50589
Street A

Two columns have only one unique value: `Status` (always `"Completed"`) and `Type of Service Request` (always `"Abandoned Vehicle Complaint"`). These columns carry zero information — every row has the same value — so they can be dropped. This is a type of data quality problem that `info()` alone would not reveal.

## Detailed missing-value audit

In [5]:
n = len(raw)
missing = raw.isnull().sum()
pct = (missing / n * 100).round(1)
audit = pd.DataFrame({"missing": missing, "pct": pct})
audit[audit["missing"] > 0].sort_values("pct", ascending=False)

,missing,pct
SSA,65301,87.1
How Many Days Has the Vehicle Been Reported as Parked?,4989,6.7
Vehicle Make/Model,3078,4.1
Most Recent Action,2359,3.1
Current Activity,2231,3.0
License Plate,2069,2.8
Vehicle Color,1491,2.0
ZIP Code,539,0.7
Police District,75,0.1
Ward,75,0.1


`SSA` is missing for 87% of rows. A column that is empty for nine out of ten rows is almost never useful for analysis and should be dropped. `Current Activity` and `Most Recent Action` appear to be entirely null (if they show 100% missing). `ZIP Code` is missing for less than 1% of rows, which is small enough to handle by dropping those specific rows.

## Step 1: Rename columns

Column names with spaces, slashes, and question marks require bracket notation everywhere: `df["How Many Days Has the Vehicle Been Reported as Parked?"]`. Converting to snake_case lets you use dot notation and makes the code readable.

In [6]:
column_map = {
    "Creation Date":                                        "creation_date",
    "Status":                                               "status",
    "Completion Date":                                      "completion_date",
    "Service Request Number":                               "request_number",
    "Type of Service Request":                              "request_type",
    "License Plate":                                        "license_plate",
    "Vehicle Make/Model":                                   "make_model",
    "Vehicle Color":                                        "color",
    "Current Activity":                                     "current_activity",
    "Most Recent Action":                                   "recent_action",
    "How Many Days Has the Vehicle Been Reported as Parked?": "days_parked",
    "Street Address":                                       "street_address",
    "ZIP Code":                                             "zip_code",
    "Ward":                                                 "ward",
    "Police District":                                      "police_district",
    "Community Area":                                       "community_area",
    "SSA":                                                  "ssa"
}

df = raw.rename(columns=column_map)
df.columns.tolist()

['creation_date',
 'status',
 'completion_date',
 'request_number',
 'request_type',
 'license_plate',
 'make_model',
 'color',
 'current_activity',
 'recent_action',
 'days_parked',
 'street_address',
 'zip_code',
 'ward',
 'police_district',
 'community_area',
 'ssa']

All column names are now lowercase with underscores. Every subsequent cell in this notebook can use dot notation (`df.zip_code`) or simple bracket notation (`df["zip_code"]`) without escaping special characters.

## Step 2: Drop columns that carry no information

Remove `status` and `request_type` (single-value columns) and `ssa` (87% missing). Also drop `current_activity` and `recent_action`, which appear to be entirely null.

In [7]:
drop_cols = ["status", "request_type", "ssa", "current_activity", "recent_action"]
df = df.drop(columns=drop_cols)
print(f"Columns remaining: {df.shape[1]}")
df.columns.tolist()

Columns remaining: 12


['creation_date',
 'completion_date',
 'request_number',
 'license_plate',
 'make_model',
 'color',
 'days_parked',
 'street_address',
 'zip_code',
 'ward',
 'police_district',
 'community_area']

Down from 17 columns to 12. Every remaining column contains potentially useful information.

## Step 3: Parse dates

Both date columns are stored as strings in `MM/DD/YYYY` format. Converting them to `datetime64` enables filtering by date range and computing the number of days to resolve each complaint.

In [8]:
df["creation_date"]   = pd.to_datetime(df["creation_date"],   format="%m/%d/%Y")
df["completion_date"] = pd.to_datetime(df["completion_date"], format="%m/%d/%Y")

print("creation_date dtype: ",  df["creation_date"].dtype)
print("completion_date dtype:", df["completion_date"].dtype)
df[["creation_date", "completion_date"]].head(3)

creation_date dtype:  datetime64[us]
completion_date dtype: datetime64[us]


,creation_date,completion_date
0,2011-01-01,2011-01-05
1,2011-01-01,2011-01-05
2,2011-01-01,2011-01-05


Both columns are now `datetime64`. With proper datetime columns, we can compute the resolution time for each complaint.

In [9]:
df["resolution_days"] = (df["completion_date"] - df["creation_date"]).dt.days
print("Resolution time (days):")
print(df["resolution_days"].describe().round(1))

Resolution time (days):
count    74999.0
mean        21.5
std         49.3
min          0.0
25%          6.0
50%         16.0
75%         30.0
max       3431.0
Name: resolution_days, dtype: float64


The median resolution time tells us how long a typical complaint took to resolve. This derived column was impossible to compute before the date columns were parsed.

## Step 4: Handle remaining missing values

With the obviously bad columns gone, check what is still missing.

In [10]:
missing_remaining = df.isnull().sum()
missing_remaining[missing_remaining > 0]

license_plate      2069
make_model         3078
color              1491
days_parked        4989
street_address       15
zip_code            539
ward                 75
police_district      75
community_area       95
dtype: int64

The remaining missing values fall into two categories: vehicle-specific fields (`license_plate`, `make_model`, `color`) that are missing because not every record had vehicle details entered, and geographic fields (`zip_code`, `ward`, `police_district`, `community_area`) that are missing for a small number of rows.

For the vehicle fields, missing means the information was not available — we should leave them as `NaN` rather than inventing values. For the geographic fields with very few missing rows, we can drop those rows without meaningfully affecting the dataset.

In [11]:
before = len(df)
df = df.dropna(subset=["zip_code", "ward", "police_district", "community_area", "street_address"])
after = len(df)
print(f"Dropped {before - after} rows with missing geographic data")
print(f"Rows remaining: {after}")

Dropped 593 rows with missing geographic data
Rows remaining: 74406


We dropped a small number of rows where the location information was incomplete. The analysis still has nearly all of the original data.

## Step 5: Normalize strings

Street addresses are in all-caps, and vehicle colors might have inconsistent capitalization if the data came from different sources. Converting to title case makes the data more readable and consistent.

In [12]:
print("Before: ", df["street_address"].head(3).tolist())

df["street_address"] = df["street_address"].str.strip().str.title()
df["color"]          = df["color"].str.strip().str.title()
df["make_model"]     = df["make_model"].str.strip().str.title()

print("After:  ", df["street_address"].head(3).tolist())

Before:  ['6059 S KOMENSKY AVE', '4651 S WASHTENAW AVE', '6200 S MASSASOIT AVE']


After:   ['6059 S Komensky Ave', '4651 S Washtenaw Ave', '6200 S Massasoit Ave']


The addresses are now in title case. The same normalization applies to color and make/model, which ensures that `"WHITE"`, `"White"`, and `" white "` all become `"White"` before we count or group by color.

## Step 6: Check for duplicates

In [13]:
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

# The request_number should be unique per complaint
n_dupes_id = df.duplicated(subset=["request_number"]).sum()
print(f"Duplicate request numbers: {n_dupes_id}")

Duplicate rows: 0
Duplicate request numbers: 502


No duplicate rows and no duplicate request numbers. Each row represents a distinct service request.

## Step 7: Validate the result

Before declaring the data clean, run one final check: compare the before and after states side by side.

In [14]:
print("=== BEFORE ===")
print(f"Shape: {raw.shape}")
print(f"Null counts: {raw.isnull().sum().sum()} total nulls")
print()
print("=== AFTER ===")
print(f"Shape: {df.shape}")
print(f"Null counts: {df.isnull().sum().sum()} total nulls")
print()
print("Remaining dtypes:")
print(df.dtypes)

=== BEFORE ===
Shape: (74999, 17)
Null counts: 82317 total nulls

=== AFTER ===
Shape: (74406, 13)
Null counts: 11477 total nulls

Remaining dtypes:
creation_date      datetime64[us]
completion_date    datetime64[us]
request_number                str
license_plate                 str
make_model                    str
color                         str
days_parked               float64
street_address                str
zip_code                  float64
ward                      float64
police_district           float64
community_area            float64
resolution_days             int64
dtype: object


The dataset went from 17 columns to 13, and most of the nulls were accounted for by dropping uninformative columns. The date columns are now `datetime64`. The derived `resolution_days` column is new.

## A quick analysis to confirm the data is ready

The best test that a dataset is truly clean is to run the analysis you actually wanted to do. Here is a groupby that would have been impossible on the raw data.

In [15]:
# Average resolution time by vehicle color (top 8 most common)
top_colors = df["color"].value_counts().head(8).index
(
    df[df["color"].isin(top_colors)]
    .groupby("color")["resolution_days"]
    .agg(["mean", "median", "count"])
    .rename(columns={"mean": "avg_days", "median": "med_days", "count": "n"})
    .sort_values("avg_days")
    .round(1)
)

,avg_days,med_days,n
color,,,
Gray,19.9,15.0,7488
Black,20.1,16.0,11793
White,20.1,16.0,12939
Blue,20.5,16.0,8176
Red,20.6,16.0,6892
Silver,20.8,16.0,5445
Green,21.7,17.0,6663
Tan,21.9,17.0,2222


This analysis required every cleaning step we ran: the column rename made `color` and `resolution_days` accessible by name, the date parsing made `resolution_days` computable, and the string normalization unified color values so that grouping works correctly.

## Packaging it as a function

The cleaning steps above work fine as a sequence of cells. But if you need to apply the same cleaning to an updated version of the data, or share it with a collaborator, a function is much better. A cleaning function takes the raw DataFrame as input and returns the clean DataFrame, with every step documented in the body.

In [16]:
def clean_311(raw: pd.DataFrame) -> pd.DataFrame:
    column_map = {
        "Creation Date":                                          "creation_date",
        "Status":                                                 "status",
        "Completion Date":                                        "completion_date",
        "Service Request Number":                                 "request_number",
        "Type of Service Request":                                "request_type",
        "License Plate":                                          "license_plate",
        "Vehicle Make/Model":                                     "make_model",
        "Vehicle Color":                                          "color",
        "Current Activity":                                       "current_activity",
        "Most Recent Action":                                     "recent_action",
        "How Many Days Has the Vehicle Been Reported as Parked?": "days_parked",
        "Street Address":                                         "street_address",
        "ZIP Code":                                               "zip_code",
        "Ward":                                                   "ward",
        "Police District":                                        "police_district",
        "Community Area":                                         "community_area",
        "SSA":                                                    "ssa",
    }

    df = raw.rename(columns=column_map)

    # Drop columns that carry no information or are mostly empty
    df = df.drop(columns=["status", "request_type", "ssa",
                           "current_activity", "recent_action"])

    # Parse dates
    df["creation_date"]   = pd.to_datetime(df["creation_date"],   format="%m/%d/%Y")
    df["completion_date"] = pd.to_datetime(df["completion_date"], format="%m/%d/%Y")

    # Derived column: days to resolve
    df["resolution_days"] = (df["completion_date"] - df["creation_date"]).dt.days

    # Drop rows with missing geographic data (small number)
    df = df.dropna(subset=["zip_code", "ward", "police_district",
                            "community_area", "street_address"])

    # Normalize strings
    for col in ["street_address", "color", "make_model"]:
        df[col] = df[col].str.strip().str.title()

    return df.reset_index(drop=True)


# Apply to the original raw data
df_clean = clean_311(raw)
print(f"Clean shape: {df_clean.shape}")
df_clean.head(3)

Clean shape: (74406, 13)


,creation_date,completion_date,request_number,license_plate,make_model,color,days_parked,street_address,zip_code,ward,police_district,community_area,resolution_days
0,2011-01-01,2011-01-05,11-00001976,H924236,Ford,White,60.0,6059 S Komensky Ave,60629.0,13.0,8.0,65.0,4
1,2011-01-01,2011-01-05,11-00002291,810 LYB WISCONSIN PLATES,Mercury,Green,NaN,4651 S Washtenaw Ave,60632.0,12.0,9.0,58.0,4
2,2011-01-01,2011-01-05,11-00002696,368M783,Buick,Gold,10.0,6200 S Massasoit Ave,60638.0,13.0,8.0,64.0,4


The function encapsulates every cleaning decision in one place. Anyone reading the function can see exactly what was dropped and why, what format the dates use, and which columns get normalized. When the upstream data changes, you run `clean_311(new_raw)` and get a clean DataFrame immediately.

## What's next

You have now seen every tool in the module: missing data, type conversions, string cleaning, text splitting and extraction, regular expressions, date parsing, and a complete pipeline. Notebook 08.9 is a set of exercises that let you practice each of these tools independently.